# Word2Vec

## Setup and Imports

In [2]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [4]:
import gensim
gensim.__version__

'4.3.3'

In [5]:
OHCO = ['doc_title','para_num','sentence_num','token_num']
BAG = OHCO[:3]


In [6]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna() # Dropped nan tokens because it was causing errors in embeding
TOKENS.head()

pos_tuple pos token_str  \
doc_title para_num sentence_num token_num                                 
ASHPUTTEL 0        0            0           ('The', 'DT')  DT       The   
                                1          ('wife', 'NN')  NN      wife   
                                2            ('of', 'IN')  IN        of   
                                3             ('a', 'DT')  DT         a   
                                4          ('rich', 'JJ')  JJ      rich   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ

## Convert to Gensim

In [7]:
docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()

In [8]:
for i in range(5):
    print(f"Doc {i}:", docs[i])

Doc 0: ['the', 'wife', 'of', 'a', 'rich', 'man', 'fell', 'sick']
Doc 1: ['and', 'when', 'she', 'felt', 'that', 'her', 'end', 'drew', 'nigh', 'she', 'called', 'her', 'only', 'daughter', 'to', 'her', 'bedside', 'and', 'said', 'always', 'be', 'a', 'good', 'girl', 'and', 'i', 'will', 'look', 'down', 'from', 'heaven', 'and', 'watch', 'over', 'you']
Doc 2: ['soon', 'afterwards', 'she', 'shut', 'her', 'eyes', 'and', 'died', 'and', 'was', 'buried', 'in', 'the', 'garden']
Doc 3: ['and', 'the', 'little', 'girl', 'went', 'every', 'day', 'to', 'her', 'grave', 'and', 'wept', 'and', 'was', 'always', 'good', 'and', 'kind', 'to', 'all', 'about', 'her']
Doc 4: ['and', 'the', 'snow', 'fell', 'and', 'spread', 'a', 'beautiful', 'white', 'covering', 'over', 'the', 'grave']


In [9]:
dictionary = Dictionary(docs) 


## Generate Embeddings

In [10]:
w2v_params = dict(
    window = 2,
    vector_size = 200,
    min_count = 50, 
    workers = 4
)

In [11]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

array([[-0.12565742,  0.12175192, -0.04597675, ..., -0.22109428,
        -0.09280045, -0.169264  ],
       [-0.10316016,  0.09518283, -0.08076137, ..., -0.22239652,
        -0.05942689, -0.1622045 ],
       [-0.01774144,  0.00562902, -0.06689824, ..., -0.21914096,
         0.00700134, -0.10428248],
       ...,
       [-0.01513456,  0.00336211, -0.05796928, ..., -0.2083112 ,
        -0.0025242 , -0.08816522],
       [-0.01436562,  0.00049905, -0.05769907, ..., -0.19712941,
        -0.01261551, -0.08306472],
       [ 0.02241575, -0.04003379, -0.05971145, ..., -0.19736116,
         0.00990569, -0.05644853]], dtype=float32)

In [12]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.125657,0.121752,-0.045977,0.045283,0.023732,0.028099,0.165081,0.246954,-0.040568,0.173077,...,0.098821,0.044967,-0.105641,-0.233657,0.158866,-0.004274,0.125733,-0.221094,-0.092800,-0.169264
and,-0.103160,0.095183,-0.080761,0.073237,0.069189,0.000929,0.147214,0.256990,-0.063626,0.161435,...,0.085796,0.049035,-0.124748,-0.224548,0.167153,0.008698,0.114752,-0.222397,-0.059427,-0.162205
to,-0.017741,0.005629,-0.066898,0.131849,0.119208,-0.072569,0.099202,0.233626,-0.070596,0.122830,...,0.073856,0.041784,-0.098297,-0.165746,0.129802,0.005396,0.101429,-0.219141,0.007001,-0.104282
he,-0.036636,0.035190,-0.080545,0.092477,0.095754,-0.032855,0.140334,0.216126,-0.040207,0.127736,...,0.100841,0.016576,-0.104455,-0.164673,0.156014,0.030807,0.094190,-0.234406,-0.009858,-0.136028
a,-0.054364,0.044088,-0.041180,0.135515,0.113593,-0.047159,0.147870,0.248355,-0.080550,0.131216,...,0.073081,0.027050,-0.079085,-0.141211,0.121939,-0.013412,0.089287,-0.187048,-0.015123,-0.104156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wished,-0.007785,-0.011364,-0.059116,0.142236,0.121098,-0.076870,0.094017,0.219103,-0.054026,0.096945,...,0.053327,0.017591,-0.062761,-0.120615,0.091588,0.006926,0.082148,-0.174130,-0.004618,-0.079605
better,0.025320,-0.041213,-0.065581,0.196692,0.176786,-0.128106,0.104107,0.255790,-0.076916,0.090410,...,0.057547,0.005963,-0.064101,-0.112392,0.091897,0.016927,0.096787,-0.210314,0.017458,-0.067696
third,-0.015135,0.003362,-0.057969,0.139715,0.121921,-0.075162,0.107385,0.232777,-0.060972,0.107865,...,0.068694,0.012687,-0.074624,-0.142055,0.116626,0.010334,0.090242,-0.208311,-0.002524,-0.088165


## Make and Plot TSNE

In [13]:
PP = 50 #40 # Try 1, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

,x,y
term_str,,
the,2.819780,13.437757
and,2.649712,12.904944
to,4.182032,1.305838
he,4.627991,8.536207
a,0.286811,8.925768
...,...,...
wished,-2.540811,-1.603917
better,-2.214021,-8.900433
third,-0.343460,0.526388


In [14]:
fig=px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')
fig.write_image('output/p2591-WORD2VEC.png')
fig.show()

## Save Outputs

In [15]:
WV.to_csv('output/p2591-WORD2VEC.csv')
